# 01 · C: tensors in memory, a forward pass and backpropagation by hand

*Base Model Training in C, C++ and CUDA*, notebook 1 of 4

NumPy is fast because its loops are compiled C that works on plain blocks of memory. In this course you write those loops yourself: in C here, then in C++ and finally in CUDA on a graphics processor. This notebook starts from the bottom. You see what a tensor is in memory, call compiled C from Python, write the forward pass of a small network as C functions, derive and write its backward pass by hand, check every gradient against finite differences, and train the network with a loop that runs entirely in C.

## What you will build

- `row_major_strides(shape)`: where each element of a tensor lives in a flat block of memory;
- `relu_forward` and `matmul` in C: your first compiled functions, with a check that runs them in a separate process;
- `log_softmax_rows` in C: the numerically stable log-softmax that classification losses need;
- the backward rules `matmul_backward`, `relu_backward` and `softmax_ce_backward` in C;
- `compute_gradients`: the full forward and backward pass of a 2 → 16 → 2 network, which a C loop then trains.

## Learning objectives

By the end of this notebook you will be able to:

1. explain how a matrix is stored in row-major order, compute an element's offset from its index and strides, and pass a buffer and its shape to C;
2. compile C into a shared library, load it with `ctypes`, and declare the argument and return types that cross the boundary;
3. write matrix multiplication, a stable log-softmax and their gradients as loops over raw memory, and explain why loop order changes speed;
4. derive the gradients of a dense layer, ReLU and softmax cross-entropy, and verify them with central differences;
5. train a small network from Python through a C training step, and recognise common C memory errors with AddressSanitizer.

### How to work through this notebook

Run the cells from top to bottom. At each **✍️ Your turn** you write C code inside the string that a small Python function returns: delete the line `raise NotImplementedError`, type your code where the comment says, and run the check cell beneath it. The check compiles your code into a small test program, runs it in a separate process and compares what it prints with what the reference solution prints:

- ✅ means your code matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not compile or does not match yet. The check shows the compiler's message or the first difference. In both cases the notebook carries on with a hidden reference version, so *Run All* always works and you can come back later.

Because each check runs in its own process, a crash or an endless loop in your C code fails the check instead of stopping the notebook. Compiler messages name your function, as in `your matmul:5:13`, and count lines from the first line of your code.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 0 for Python:** the notebook's Python uses only the standard library. `ctypes` loads compiled code and calls it, `subprocess` runs the compiler and the check programs, and `array` holds numbers in plain C memory. NumPy is not imported anywhere: every calculation of the model happens in C that you write.
- **A C compiler:** gcc or clang, using the C11 standard. The course notebook server has one, and so does Google Colab. On your own computer, Linux and macOS (with the Xcode command-line tools) work; on Windows, use WSL.
- **Not in the browser preview:** JupyterLite runs Python inside the web page and cannot run a compiler, so open this notebook on the notebook server, in Colab or locally.
- Nothing is downloaded, and every random number comes from a seeded generator.
- You should know the forward and backward pass of a small network, as built in *Base Model Training* or *Neural Networks from Scratch*. C syntax is explained as it appears, but it helps to have seen a C `for` loop and a pointer before.

## Setup

Run this cell first. It finds the C and C++ compilers and defines the tools the notebook uses:

- `build(source, language)` compiles source code into a program, and `load(source)` compiles it into a shared library and loads it into Python;
- `run(program)` runs a compiled program in its own process and returns what it printed;
- `mark(source, label)` splices code into a larger program so that compiler messages name `label`;
- `practice_native` and `practice` check what you write against the reference;
- `address(buffer)` gives the memory address of an `array`, which is how Python hands a buffer to C.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import atexit
import ctypes
import hashlib
import math
import os
import random
import shutil
import signal
import subprocess
import sys
import tempfile
from array import array

if sys.platform == "emscripten":
    raise RuntimeError(
        "This notebook compiles C, C++ and CUDA, which a web browser cannot do. Open it on your "
        "course notebook server, in Google Colab, or in Jupyter on a computer with a C compiler."
    )

CC = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
CXX = shutil.which("c++") or shutil.which("g++") or shutil.which("clang++")
if not (CC and CXX):
    raise RuntimeError("No C and C++ compiler found: install gcc and g++ (or clang), then restart the kernel.")
BUILD_DIR = tempfile.mkdtemp(prefix="furina-native-")
atexit.register(shutil.rmtree, BUILD_DIR, True)
COMPILED = {}  # source hash -> path of the program or library built from it


class CompileError(Exception):
    """The compiler rejected the code. The message is what the compiler printed."""


class RunError(Exception):
    """A compiled program crashed, never finished, or stopped with an error."""


def _trim(text, lines=14):
    kept = text.strip().splitlines()
    return "\n".join(kept[:lines] + (["   ..."] if len(kept) > lines else []))


def mark(source, label):
    """Splice in source so that compiler messages name `label` and count from its first line."""
    return f'\n#line 1 "{label}"\n{source}\n'


def build(source, language="c", shared=False, flags=()):
    """Compile C ("c"), C++ ("c++") or CUDA ("cuda") into a program, or a shared library."""
    key = hashlib.sha256(f"{language}|{shared}|{flags}|{source}".encode()).hexdigest()[:24]
    if key in COMPILED:
        return COMPILED[key]
    output = os.path.join(BUILD_DIR, key + (".so" if shared else ".exe"))
    if language == "cuda":
        command = cuda_command(source, key, output)  # defined in the CUDA notebooks
    else:
        path = os.path.join(BUILD_DIR, key + (".c" if language == "c" else ".cpp"))
        with open(path, "w", encoding="utf-8") as handle:
            handle.write(source)
        if language == "c":
            command = [CC, "-std=c11", "-O2", "-ffp-contract=off", "-Wall", path, "-o", output, "-lm"]
        else:
            command = [CXX, "-std=c++17", "-O2", "-ffp-contract=off", "-Wall", path, "-o", output]
    if shared:
        command[1:1] = ["-shared"] + (["-Xcompiler", "-fPIC"] if command[0] == globals().get("NVCC") else ["-fPIC"])
    command += list(flags)  # last, so that a flag such as -O1 overrides the default
    result = subprocess.run(command, capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        raise CompileError(_trim(result.stderr or result.stdout))
    COMPILED[key] = output
    return output


def load(source, language="c"):
    """Compile source into a shared library and load it into this Python process."""
    return ctypes.CDLL(build(source, language, shared=True))


def run(program, timeout=20, env=None):
    """Run a compiled program in its own process and return what it printed."""
    try:
        result = subprocess.run([program], capture_output=True, text=True, timeout=timeout,
                                env=None if env is None else {**os.environ, **env})
    except subprocess.TimeoutExpired:
        raise RunError(f"did not finish within {timeout} seconds: is there a loop that never ends?") from None
    if result.returncode < 0:
        try:
            name = signal.Signals(-result.returncode).name
        except ValueError:
            name = f"signal {-result.returncode}"
        hint = {
            "SIGSEGV": "a read or write outside memory the program owns, such as an index past the end of an array",
            "SIGBUS": "a read or write outside memory the program owns",
            "SIGFPE": "an integer division by zero",
            "SIGABRT": "the program aborted itself, often after finding corrupted memory",
            "SIGTRAP": "the compiler turned undefined behaviour, such as using a null pointer, into a trap",
            "SIGILL": "the compiler turned undefined behaviour, such as using a null pointer, into a trap",
        }.get(name, "")
        detail = f"\n{_trim(result.stderr)}" if result.stderr.strip() else ""
        raise RunError(f"crashed with {name}" + (f" ({hint})" if hint else "") + detail)
    if result.returncode != 0:
        raise RunError(f"stopped with exit status {result.returncode}:\n{_trim(result.stderr + result.stdout)}")
    return result.stdout


def _value(token):
    try:
        return float(token)
    except ValueError:
        return None


def _same(a, b, atol, rtol):
    if a == b:
        return True
    x, y = _value(a), _value(b)
    if x is None or y is None:
        return False
    if math.isnan(x) or math.isnan(y):
        return math.isnan(x) and math.isnan(y)
    return math.isclose(x, y, rel_tol=rtol, abs_tol=atol)


def _short(text, limit=110):
    text = " ".join(text.split())
    return text if len(text) <= limit else text[: limit - 3] + "..."


def compare_output(got, want, atol=1e-9, rtol=1e-9):
    """None when two printouts match line by line, otherwise the first difference."""
    got_lines, want_lines = got.strip().splitlines(), want.strip().splitlines()
    for got_line, want_line in zip(got_lines, want_lines):
        got_label, _, got_values = got_line.partition(":")
        want_label, _, want_values = want_line.partition(":")
        g, w = got_values.split(), want_values.split()
        if got_label != want_label or len(g) != len(w) or not all(_same(a, b, atol, rtol) for a, b in zip(g, w)):
            return (f"for {want_label.strip() or 'a case'}\n   your code printed {_short(got_values) or '(nothing)'}"
                    f"\n   expected          {_short(want_values)}")
    if len(got_lines) < len(want_lines):
        return f"your program stopped before printing {want_lines[len(got_lines)].partition(':')[0]!r}"
    if len(got_lines) > len(want_lines):
        return f"your program printed {len(got_lines) - len(want_lines)} line(s) more than expected"
    return None


REFERENCE_OUTPUT = {}


def practice_native(name, yours, reference, program, language="c", atol=1e-9, rtol=1e-9, timeout=20):
    """Check native code you wrote against the reference.

    `yours` and `reference` return source code. `program(source)` wraps a
    source in a complete program whose main() runs the cases and prints the
    results. Both programs are compiled and run in their own processes, so a
    crash or an endless loop fails the check without stopping this notebook.
    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference, so you can always keep going.
    """
    fallback = "   The rest of the notebook uses the reference version for now."
    key = hashlib.sha256(f"{language}|{program(reference())}".encode()).hexdigest()
    if key not in REFERENCE_OUTPUT:
        REFERENCE_OUTPUT[key] = run(build(program(reference()), language), timeout)
    try:
        source = yours()
        if not isinstance(source, str):
            raise TypeError(f"{name} must return its source code as a string, not {type(source).__name__}")
        got = run(build(program(source), language), timeout)
    except NotImplementedError:
        print(f"✍️ {name}: not written yet.")
        print(fallback)
        return reference
    except CompileError as error:
        print(f"❌ {name}: the compiler rejected your code:\n{error}")
        print(fallback)
        return reference
    except RunError as error:
        print(f"❌ {name}: the check program {error}")
        print(fallback)
        return reference
    except Exception as error:  # a mistake in the Python around the source
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(fallback)
        return reference
    difference = compare_output(got, REFERENCE_OUTPUT[key], atol, rtol)
    if difference:
        print(f"❌ {name}: {difference}")
        print(fallback)
        return reference
    print(f"✅ {name}: matches the reference. From here on, the notebook uses your version.")
    return yours


def _close(got, want, atol):
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want))
    if isinstance(want, float) or isinstance(got, float):
        try:
            return math.isclose(float(got), float(want), rel_tol=1e-9, abs_tol=atol)
        except (TypeError, ValueError):
            return False
    return got == want


def practice(name, yours, reference, cases, atol=1e-9):
    """Check a Python function you wrote against the reference on a few inputs."""
    fallback = "   The rest of the notebook uses the reference version for now."
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(repr(args))}")
                print(f"   you returned {_short(repr(got))}")
                print(f"   expected     {_short(repr(want))}")
                print(fallback)
                return reference
    except NotImplementedError:
        print(f"✍️ {name}: not written yet.")
        print(fallback)
        return reference
    except Exception as error:
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(fallback)
        return reference
    print(f"✅ {name}: matches the reference. From here on, the notebook uses your version.")
    return yours


def address(buffer):
    """The memory address of an array.array, to pass to C as a pointer."""
    return buffer.buffer_info()[0]


def c_list(values, kind="double"):
    """Python numbers as a C initialiser list, such as {1.5, -2.0}."""
    if kind == "int":
        return "{" + ", ".join(str(int(v)) for v in values) + "}"
    suffix = "f" if kind == "float" else ""
    return "{" + ", ".join(f"{float(v)!r}{suffix}" for v in values) + "}"


print(f"C compiler: {CC}\nC++ compiler: {CXX}")

The check programs share a few lines of C: the headers, `show`, which prints an array on one labelled line with 17 significant digits (enough to reproduce a `double` exactly), and `fill`, which writes the same test numbers on every machine. Each check program is this prelude, then your code, then a `main` that calls your function on a few cases.

In [ ]:
CHECK_C = r"""
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

static void show(const char *label, const double *x, long n)
{
    printf("%s:", label);
    for (long i = 0; i < n; i++)
        printf(" %.17g", x[i]);
    printf("\n");
}

/* Deterministic test numbers: entry i is sin(seed + 0.37 i). */
static void fill(double *x, long n, double seed)
{
    for (long i = 0; i < n; i++)
        x[i] = sin(seed + 0.37 * (double)i);
}
"""

## 1 · A tensor is a block of memory and a shape

A NumPy array, a PyTorch tensor and the matrices in this course all store their numbers the same way: in **one contiguous block of memory**, plus a description of how to find element $(i, j)$ in it. C sees only the block. A `double *` points at the first number, and the shape travels separately, as plain integers.

In **row-major** order, which C, NumPy and PyTorch use by default, the rows are laid end to end. For an $n 	imes m$ matrix, element $(i, j)$ sits at

$$\text{offset}(i, j) = i \cdot m + j,$$

counted in elements, not bytes. The numbers $(m, 1)$ are the **strides**: how far the offset moves when $i$ or $j$ grows by one. In general, for an index $(i_0, i_1, \ldots)$ and strides $(s_0, s_1, \ldots)$, the offset is $\sum_k i_k s_k$, and in row-major order the last axis has stride 1 and each earlier stride is the product of the sizes after it.

Strides explain why some operations are free. Reading the same block with strides $(1, m)$ instead of $(m, 1)$ walks it column by column, which is the **transpose**, and no number moves. Frameworks call such a reinterpretation a *view*. The price is that a transposed view is no longer contiguous along its rows, so code that assumes stride 1 along the last axis needs a copy first.

In [ ]:
flat = array("d", [float(v) for v in range(12)])    # 12 doubles in one block: a 3 x 4 matrix in row-major order


def element(buffer, index, strides):
    return buffer[sum(i * s for i, s in zip(index, strides))]


print("row 1 of the 3 x 4 matrix:", [element(flat, (1, j), (4, 1)) for j in range(4)])
print("row 1 of its 4 x 3 transpose, read through strides (1, 4):", [element(flat, (1, j), (1, 4)) for j in range(3)])
print("bytes per double:", ctypes.sizeof(ctypes.c_double))
start = address(flat)
print(f"element 0 is at address {start:#x}, element 5 at {start + 5 * 8:#x}: 5 x 8 bytes further on")

### ✍️ Your turn: `row_major_strides`

Write `row_major_strides(shape)`, a warm-up in Python:

- **input:** a tuple of axis sizes, such as `(3, 4)` or `(2, 3, 4)`;
- **output:** the row-major strides in elements, as a tuple: `(4, 1)` and `(12, 4, 1)` for those shapes. The last stride is 1, and each earlier stride is the product of all the sizes after it.

Hint: walk the shape backwards with a running product, then reverse the list you built. An empty shape (a single number) has empty strides.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def row_major_strides_reference(shape):
    strides, step = [], 1
    for size in reversed(shape):
        strides.append(step)
        step *= size
    return tuple(reversed(strides))

In [ ]:
def row_major_strides(shape):
    """Row-major strides, in elements, for a tensor of this shape: (3, 4) -> (4, 1)."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
row_major_strides = practice("row_major_strides", row_major_strides, row_major_strides_reference, cases=[
    ((3, 4),),
    ((2, 3, 4),),
    ((5,),),
    ((),),            # edge case: a single number has no axes
])

**Expected behaviour.** `(3, 4)` gives `(4, 1)`, `(2, 3, 4)` gives `(12, 4, 1)`, a vector of 5 gives `(1,)` and the empty shape gives `()`. With strides you can find any element of a contiguous tensor from a single pointer, which is all a C function receives.

## 2 · Calling C from Python

A C function is compiled ahead of time into machine code. Python can call it through `ctypes` once three things are settled:

1. **Compile into a shared library.** `load(source)` runs the C compiler with `-shared -fPIC`, which produces a `.so` file of machine code that a running program can load, and `ctypes.CDLL` loads it.
2. **Declare the types.** C functions carry no type information at run time, so Python must say what each argument is. Here a pointer is passed as `ctypes.c_void_p` (an address), a C `long` as `ctypes.c_long` and a `double` as `ctypes.c_double`. Set `restype` too: `ctypes` assumes a C `int` result unless told otherwise, which would silently cut a returned pointer or `double` to the wrong type.
3. **Hand over memory.** An `array("d", ...)` keeps its numbers in one contiguous block of C `double`s, so its address is exactly the `double *` a C function expects. Python still owns the memory: C borrows it for the duration of the call, and must not keep the pointer afterwards. Keep the array in a variable while C uses it. In `f(address(array("d", values)))` the temporary array can be freed as soon as `address` returns, before `f` reads the memory, which hands C garbage.

The function below scales a vector in place. Read it, run the cell, and match each `ctypes` declaration with the C signature.

In [ ]:
SCALE_C = r"""
void scale(double *x, long n, double factor)
{
    for (long i = 0; i < n; i++)
        x[i] *= factor;
}
"""
demo = load(SCALE_C)
demo.scale.argtypes = [ctypes.c_void_p, ctypes.c_long, ctypes.c_double]
demo.scale.restype = None                  # a C function returning void

values = array("d", [1.0, -2.0, 3.5])
demo.scale(address(values), len(values), 10.0)
print("after scale(x, 3, 10.0):", list(values))

**What to notice.** The C loop changed Python's `array` in place, because both sides used the same memory. Nothing checked that `n` matched the array's length: pass `n = 300` and C would read and write past the end of the block, into whatever memory follows. C trusts the sizes it is given, which is why every function in this course takes its sizes explicitly and every check tests them.

## 3 · Your first kernel: ReLU

ReLU keeps positive numbers and replaces the rest with zero: $\operatorname{relu}(x) = \max(x, 0)$. On a matrix it works element by element, so the C function does not need the shape at all, only the total number of elements: a $32 \times 16$ matrix is 512 doubles in a row.

A few pieces of C syntax you need:

- `const double *x` is a pointer to doubles that the function promises not to change; `double *y` may be written;
- `x[i]` reads the element `i` places after the one `x` points to, exactly like `*(x + i)`;
- `for (long i = 0; i < n; i++) { ... }` counts `i` from 0 to `n - 1`, and `long` is a 64-bit integer on Linux and macOS, large enough for any index;
- `condition ? a : b` is C's conditional expression, the same as Python's `a if condition else b`.

### ✍️ Your turn: `relu_forward`

Complete the C function `relu_forward(x, y, n)`:

- **inputs:** `x`, a pointer to `n` doubles; `y`, a pointer to `n` doubles to write; `n`, the number of elements;
- **effect:** `y[i]` becomes `x[i]` when `x[i] > 0`, and `0.0` otherwise, for every `i` from 0 to `n - 1`. It returns nothing.

Hint: one `for` loop with one assignment. Delete the `raise NotImplementedError` line before you run the check.

*Type it rather than pasting it: typing is how the syntax sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def relu_forward_c_reference():
    return r"""
void relu_forward(const double *x, double *y, long n)
{
    for (long i = 0; i < n; i++)
        y[i] = x[i] > 0.0 ? x[i] : 0.0;
}
"""

In [ ]:
def relu_forward_c():
    raise NotImplementedError  # ✍️ delete this line, then complete the C function below
    return r"""
void relu_forward(const double *x, double *y, long n)
{
    /* ✍️ Your code here (one loop) */
}
"""

In [ ]:
def relu_program(source):
    return CHECK_C + mark(source, "your relu_forward") + mark(r"""
int main(void)
{
    double x[] = {-2.0, -0.5, 0.0, 0.5, 3.0}, y[5];
    relu_forward(x, y, 5);
    show("mixed signs", y, 5);

    double big[] = {-1e300, 1e300, -1e-300, 1e-300}, out[4];
    relu_forward(big, out, 4);
    show("very large and very small", out, 4);

    double untouched[] = {7.0};
    relu_forward(x, untouched, 0);          /* n = 0: nothing may be written */
    show("n is zero", untouched, 1);
    return 0;
}
""", "the check")


relu_forward_c = practice_native("relu_forward", relu_forward_c, relu_forward_c_reference, relu_program)

**Expected behaviour.** The first case prints `0 0 0 0.5 3`, and very large or very small values keep their sign rule. With `n = 0` the loop never runs and the 7 stays. If you see `your relu_forward:4:5: error: expected ';'`, the compiler is pointing at line 4 of your function: C needs a semicolon after every statement.

## 4 · Matrix multiplication

For $C = AB$ with $A$ of shape $n \times k$ and $B$ of shape $k \times m$, each output element is a dot product,

$$C_{ij} = \sum_{r=0}^{k-1} A_{ir} B_{rj},$$

which in row-major memory reads `A[i * k + r]` and `B[r * m + j]` and writes `C[i * m + j]`. A dense layer is exactly this operation followed by a bias: for a batch $X$ of shape $n \times d$, the layer computes $Z = XW + b$ with $W$ of shape $d \times h$. The batch size $n$ never appears in a weight's shape.

The function receives three pointers and the three sizes. Accumulate each dot product in a local `double`, then store it once: a local variable lives in a register, so the inner loop does not have to write to memory on every step.

### ✍️ Your turn: `matmul`

Complete `matmul(A, B, C, n, k, m)`:

- **inputs:** `A` ($n \times k$) and `B` ($k \times m$), row-major; `C`, room for $n \times m$ doubles; the sizes `n`, `k` and `m`;
- **effect:** `C[i * m + j]` becomes $\sum_r$ `A[i * k + r] * B[r * m + j]` for every `i < n` and `j < m`.

Hint: three nested loops over `i`, `j` and `r`, with `double sum = 0.0;` declared inside the `j` loop.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def matmul_c_reference():
    return r"""
void matmul(const double *A, const double *B, double *C, long n, long k, long m)
{
    for (long i = 0; i < n; i++)
        for (long j = 0; j < m; j++) {
            double sum = 0.0;
            for (long r = 0; r < k; r++)
                sum += A[i * k + r] * B[r * m + j];
            C[i * m + j] = sum;
        }
}
"""

In [ ]:
def matmul_c():
    raise NotImplementedError  # ✍️ delete this line, then complete the C function below
    return r"""
void matmul(const double *A, const double *B, double *C, long n, long k, long m)
{
    /* ✍️ Your code here (three nested loops) */
}
"""

In [ ]:
def matmul_program(source):
    return CHECK_C + mark(source, "your matmul") + mark(r"""
int main(void)
{
    double A[] = {1, 2, 3,
                  4, 5, 6};                 /* 2 x 3 */
    double B[] = {7, 8,
                  9, 10,
                  11, 12};                  /* 3 x 2 */
    double C[4];
    matmul(A, B, C, 2, 3, 2);
    show("2x3 times 3x2", C, 4);

    double a = 3.0, b = -2.0, c;
    matmul(&a, &b, &c, 1, 1, 1);
    show("1x1 times 1x1", &c, 1);

    double P[4 * 5], Q[5 * 3], R[4 * 3];
    fill(P, 20, 0.1);
    fill(Q, 15, 2.0);
    matmul(P, Q, R, 4, 5, 3);
    show("4x5 times 5x3", R, 12);

    double row[3] = {1, 2, 3}, column[3] = {4, 5, 6}, outer[9];
    matmul(column, row, outer, 3, 1, 3);    /* inner size 1: an outer product */
    show("3x1 times 1x3", outer, 9);
    return 0;
}
""", "the check")


matmul_c = practice_native("matmul", matmul_c, matmul_c_reference, matmul_program)

**Expected behaviour.** The worked case prints `58 64 139 154`: for example $C_{00} = 1 \cdot 7 + 2 \cdot 9 + 3 \cdot 11 = 58$. With an inner size of 1 the product is an outer product, a 3 × 3 table of `column[i] * row[j]`. If only the square-looking cases pass, look for a stride that uses the wrong size, such as `B[r * k + j]`.

### How much does compiling buy?

The same triple loop in Python runs one bytecode instruction at a time, and every number is a Python object. The cell below multiplies two $80 \times 80$ matrices both ways and compares the times and the results.

In [ ]:
import time

lib_mm = load(matmul_c())
lib_mm.matmul.argtypes = [ctypes.c_void_p] * 3 + [ctypes.c_long] * 3
lib_mm.matmul.restype = None

size = 80
gen = random.Random(1)
A = array("d", [gen.uniform(-1, 1) for _ in range(size * size)])
B = array("d", [gen.uniform(-1, 1) for _ in range(size * size)])
C_native = array("d", bytes(8 * size * size))      # size * size zeros

start = time.perf_counter()
lib_mm.matmul(address(A), address(B), address(C_native), size, size, size)
native_seconds = time.perf_counter() - start

start = time.perf_counter()
C_python = [sum(A[i * size + r] * B[r * size + j] for r in range(size)) for i in range(size) for j in range(size)]
python_seconds = time.perf_counter() - start

largest_gap = max(abs(p - c) for p, c in zip(C_python, C_native))
print(f"C: {native_seconds * 1e3:.2f} ms   Python: {python_seconds * 1e3:.0f} ms   "
      f"about {python_seconds / native_seconds:.0f} times faster; largest difference {largest_gap:.1e}")

**What to notice.** On a typical laptop the compiled loop is around a hundred times faster, often more, and both give the same numbers up to rounding. That gap is why numerical libraries are written in compiled languages, and why this course writes its loops in C, C++ and CUDA. Your exact ratio depends on the machine.

### 🧪 Try it: loop order and the cache

Memory is fast only near where you have just read. The processor fetches 64 bytes (8 doubles) at a time into its cache, so a loop that walks memory with stride 1 uses every number it fetched, while a loop with a large stride uses one and throws the other seven away.

In `matmul`, the innermost loop over `r` reads `B[r * m + j]`, which jumps `m` doubles on every step. Swapping the loops to the order `i`, `r`, `j` makes the innermost loop walk along a row of `B` and a row of `C` with stride 1, and does exactly the same arithmetic. Run the cell to compare both orders on a $320 \times 320$ product, then change `size` and run it again.

In [ ]:
LOOP_ORDERS_C = r"""
void matmul_ijk(const double *A, const double *B, double *C, long n, long k, long m)
{
    for (long i = 0; i < n; i++)
        for (long j = 0; j < m; j++) {
            double sum = 0.0;
            for (long r = 0; r < k; r++)
                sum += A[i * k + r] * B[r * m + j];      /* B walked down a column: stride m */
            C[i * m + j] = sum;
        }
}

void matmul_irj(const double *A, const double *B, double *C, long n, long k, long m)
{
    for (long i = 0; i < n * m; i++)
        C[i] = 0.0;
    for (long i = 0; i < n; i++)
        for (long r = 0; r < k; r++) {
            double a = A[i * k + r];
            for (long j = 0; j < m; j++)
                C[i * m + j] += a * B[r * m + j];        /* B and C walked along rows: stride 1 */
        }
}
"""
orders = load(LOOP_ORDERS_C)
for name in ("matmul_ijk", "matmul_irj"):
    getattr(orders, name).argtypes = [ctypes.c_void_p] * 3 + [ctypes.c_long] * 3
    getattr(orders, name).restype = None

size = 320  # 🧪 try 128 and 512
gen = random.Random(2)
A = array("d", [gen.uniform(-1, 1) for _ in range(size * size)])
B = array("d", [gen.uniform(-1, 1) for _ in range(size * size)])
results = {}
for name in ("matmul_ijk", "matmul_irj"):
    C = array("d", bytes(8 * size * size))
    start = time.perf_counter()
    getattr(orders, name)(address(A), address(B), address(C), size, size, size)
    results[name] = (time.perf_counter() - start, C)
    print(f"{name}: {results[name][0] * 1e3:7.1f} ms")
gap = max(abs(x - y) for x, y in zip(results["matmul_ijk"][1], results["matmul_irj"][1]))
print(f"speed-up from the loop order alone: {results['matmul_ijk'][0] / results['matmul_irj'][0]:.1f}x; "
      f"largest difference between the results {gap:.1e}")

**What to notice.** The `i, r, j` order is usually several times faster, and the gap grows with the size, once a column of `B` no longer fits in the cache. The results differ only in the last bits, because the additions happen in a different order. Optimised libraries go much further: they split the matrices into tiles that fit in the cache and use vector instructions. Notebook 3 applies the same tiling idea to a GPU's shared memory.

## 5 · Bias, log-softmax and the loss

A dense layer adds its bias $b$ (one number per output column) to every row: `Z[i * m + j] += b[j]`. Broadcasting, which NumPy does for you, is in C just a loop that reuses `b[j]` for every row `i`. The cell below defines `add_bias`, and `nll_loss`, which averages $-\log p_{i, y_i}$ over the rows, given each row's log-probabilities and the integer class labels `y`.

A classifier turns its last layer's outputs, the **logits** $z$, into probabilities with the softmax. Classification losses need their logarithm, and the stable way to compute it subtracts the row maximum $c = \max_j z_j$ first:

$$\log \operatorname{softmax}(z)_j = z_j - \log \sum_l e^{z_l} = z_j - c - \log \sum_l e^{z_l - c}.$$

Both forms are equal, but only the second is safe: $e^{1000}$ overflows a `double` to infinity, while every $e^{z_l - c}$ lies between 0 and 1 and one of them is exactly 1, so the sum is between 1 and the number of classes.

In [ ]:
ADD_BIAS_C = r"""
void add_bias(double *Z, const double *b, long n, long m)
{
    for (long i = 0; i < n; i++)
        for (long j = 0; j < m; j++)
            Z[i * m + j] += b[j];
}
"""

NLL_LOSS_C = r"""
double nll_loss(const double *logp, const int *y, long n, long m)
{
    double total = 0.0;
    for (long i = 0; i < n; i++)
        total -= logp[i * m + y[i]];        /* the log-probability of row i's true class */
    return total / n;
}
"""
print("add_bias and nll_loss are defined; the model uses them as they are.")

### ✍️ Your turn: `log_softmax_rows`

Complete `log_softmax_rows(Z, out, n, m)`:

- **inputs:** `Z`, an $n \times m$ matrix of logits; `out`, room for $n \times m$ doubles;
- **effect:** row $i$ of `out` holds the log-softmax of row $i$ of `Z`, computed with the row maximum subtracted.

Hints: for each row, set `const double *z = Z + i * m;` (a pointer to the row's first element), find the largest `z[j]`, add up `exp(z[j] - largest)`, and write `z[j] - largest - log(total)`. `exp` and `log` come from `<math.h>`, which the check includes.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def log_softmax_rows_c_reference():
    return r"""
void log_softmax_rows(const double *Z, double *out, long n, long m)
{
    for (long i = 0; i < n; i++) {
        const double *z = Z + i * m;
        double largest = z[0];
        for (long j = 1; j < m; j++)
            if (z[j] > largest)
                largest = z[j];
        double total = 0.0;
        for (long j = 0; j < m; j++)
            total += exp(z[j] - largest);
        double log_total = largest + log(total);
        for (long j = 0; j < m; j++)
            out[i * m + j] = z[j] - log_total;
    }
}
"""

In [ ]:
def log_softmax_rows_c():
    raise NotImplementedError  # ✍️ delete this line, then complete the C function below
    return r"""
void log_softmax_rows(const double *Z, double *out, long n, long m)
{
    for (long i = 0; i < n; i++) {
        const double *z = Z + i * m;
        /* ✍️ Your code here: the row maximum, the sum of exponentials, then the outputs */
    }
}
"""

In [ ]:
def log_softmax_program(source):
    return CHECK_C + mark(source, "your log_softmax_rows") + mark(r"""
int main(void)
{
    double Z[] = {1.0, 2.0, 3.0,
                  0.0, 0.0, 0.0}, out[6];
    log_softmax_rows(Z, out, 2, 3);
    show("two rows", out, 6);

    double huge[] = {1000.0, 1001.0, 1002.0}, out_huge[3];
    log_softmax_rows(huge, out_huge, 1, 3);
    show("logits near 1000", out_huge, 3);

    double tiny[] = {-1000.0, -1001.0}, out_tiny[2];
    log_softmax_rows(tiny, out_tiny, 1, 2);
    show("logits near -1000", out_tiny, 2);

    double one[] = {42.0, -7.0}, out_one[2];
    log_softmax_rows(one, out_one, 2, 1);   /* a single class: probability 1 */
    show("one column", out_one, 2);

    double R[4 * 5], out_R[4 * 5];
    fill(R, 20, 0.7);
    log_softmax_rows(R, out_R, 4, 5);
    show("4x5", out_R, 20);
    return 0;
}
""", "the check")


log_softmax_rows_c = practice_native("log_softmax_rows", log_softmax_rows_c, log_softmax_rows_c_reference,
                                     log_softmax_program)

**Expected behaviour.** The row `[1, 2, 3]` gives about `[-2.408, -1.408, -0.408]` and the uniform row gives $-\log 3 \approx -1.0986$ three times. Logits near 1000 give exactly the same answer as `[1, 2, 3]`, because only differences between logits matter. With a single column every log-probability is 0. If the case near 1000 prints `nan` or `-inf`, the row maximum was not subtracted before `exp`.

### 🧪 Try it: the unstable version

The cell below compiles the textbook formula, $\log\bigl(e^{z_j} / \sum_l e^{z_l}\bigr)$, next to your function, and runs both on logits near 1000 and on ordinary ones. Change `scale` to 1 and run it again.

In [ ]:
NAIVE_C = r"""
#include <math.h>
void log_softmax_naive(const double *Z, double *out, long n, long m)
{
    for (long i = 0; i < n; i++) {
        double total = 0.0;
        for (long j = 0; j < m; j++)
            total += exp(Z[i * m + j]);
        for (long j = 0; j < m; j++)
            out[i * m + j] = log(exp(Z[i * m + j]) / total);
    }
}
""" + "\n#include <math.h>\n" + log_softmax_rows_c()
both = load(NAIVE_C)
for name in ("log_softmax_naive", "log_softmax_rows"):
    getattr(both, name).argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_long, ctypes.c_long]
    getattr(both, name).restype = None

scale = 1000.0  # 🧪 try 1.0, then 700.0 and 710.0
logits = array("d", [scale, scale + 1.0, scale + 2.0])
for name in ("log_softmax_naive", "log_softmax_rows"):
    out = array("d", bytes(8 * 3))
    getattr(both, name)(address(logits), address(out), 1, 3)
    print(f"{name:18}: {[round(v, 4) for v in out]}")

**What to notice.** Near 1000 the naive version returns `nan`: $e^{1000}$ is infinity, and infinity divided by infinity is not a number. With `scale = 1` both agree. The threshold is $e^{709.8}$, the largest finite `double`, so 700 still works and 710 fails. A training run whose logits drift upwards would hit that threshold mid-run and fill the loss with `nan` without any error message.

## 6 · Backpropagation by hand

Frameworks derive gradients automatically; notebook 2 builds an engine that does. Here you write each gradient yourself, which is exactly what an engine's backward rules contain. Write $\bar{V} = \partial L / \partial V$ for the gradient of the loss with respect to any matrix $V$; it always has $V$'s shape.

**Matrix product.** For $C = AB$, element $A_{ir}$ affects the whole row $i$ of $C$, through $C_{ij} = \sum_r A_{ir} B_{rj}$, so its gradient sums over that row. In matrix form:

$$\bar A = \bar C B^\top \quad (n \times m)(m \times k), \qquad \bar B = A^\top \bar C \quad (k \times n)(n \times m).$$

In memory you never build the transposes: $\bar A_{ir} = \sum_j \bar C_{ij} B_{rj}$ reads row $r$ of `B`, and $\bar B_{rj} = \sum_i A_{ir} \bar C_{ij}$ reads column $r$ of `A`.

**Bias.** $b_j$ was added to every row, so $\bar b_j = \sum_i \bar Z_{ij}$, a column sum.

**ReLU.** For $H = \operatorname{relu}(Z)$, the local derivative is 1 where $Z > 0$ and 0 elsewhere, so $\bar Z = \bar H \odot [Z > 0]$. At exactly 0 the derivative is undefined; the convention, used here, is 0.

**Softmax cross-entropy.** For the mean loss $L = -\frac1n \sum_i \log p_{i, y_i}$ with $p = \operatorname{softmax}(Z)$, the gradient with respect to the logits is

$$\bar Z_{ij} = \frac{p_{ij} - [j = y_i]}{n}.$$

It is the predicted probability minus the one-hot target, divided by the batch size because the loss is a mean. Each row of $\bar Z$ therefore sums to zero.

### ✍️ Your turn: `matmul_backward`

Complete `matmul_backward(dC, A, B, dA, dB, n, k, m)` for $C = AB$ with $A$ of shape $n \times k$ and $B$ of shape $k \times m$:

- **inputs:** `dC`, the upstream gradient ($n \times m$); `A` and `B`, the matrices from the forward pass; `dA` and `dB`, room for the gradients;
- **effect:** `dA` ($n \times k$) becomes $\bar C B^\top$ and `dB` ($k \times m$) becomes $A^\top \bar C$. Either output pointer may be `NULL`, which means that gradient is not needed: skip it. The first layer's input $X$ is data, so its gradient is never needed.

Hints: `if (dA) { ... }` runs only when `dA` is not `NULL`. Each gradient is two loops over its own indices with a `sum` over the third: `dA[i * k + r]` sums `dC[i * m + j] * B[r * m + j]` over `j`, and `dB[r * m + j]` sums `A[i * k + r] * dC[i * m + j]` over `i`. Write the gradients with `=`, not `+=`: each call computes them afresh.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def matmul_backward_c_reference():
    return r"""
void matmul_backward(const double *dC, const double *A, const double *B,
                     double *dA, double *dB, long n, long k, long m)
{
    if (dA)
        for (long i = 0; i < n; i++)
            for (long r = 0; r < k; r++) {
                double sum = 0.0;
                for (long j = 0; j < m; j++)
                    sum += dC[i * m + j] * B[r * m + j];
                dA[i * k + r] = sum;
            }
    if (dB)
        for (long r = 0; r < k; r++)
            for (long j = 0; j < m; j++) {
                double sum = 0.0;
                for (long i = 0; i < n; i++)
                    sum += A[i * k + r] * dC[i * m + j];
                dB[r * m + j] = sum;
            }
}
"""

In [ ]:
def matmul_backward_c():
    raise NotImplementedError  # ✍️ delete this line, then complete the C function below
    return r"""
void matmul_backward(const double *dC, const double *A, const double *B,
                     double *dA, double *dB, long n, long k, long m)
{
    /* ✍️ Your code here: dA = dC B^T (skip when dA is NULL), then dB = A^T dC (skip when dB is NULL) */
}
"""

In [ ]:
def matmul_backward_program(source):
    return CHECK_C + mark(source, "your matmul_backward") + mark(r"""
int main(void)
{
    double A[] = {1, 2, 3, 4, 5, 6};        /* 2 x 3 */
    double B[] = {1, 0, -1, 2, 0.5, 1};     /* 3 x 2 */
    double dC[] = {1, -1, 2, 0.5};          /* 2 x 2 */
    double dA[6], dB[6];
    matmul_backward(dC, A, B, dA, dB, 2, 3, 2);
    show("dA, small", dA, 6);
    show("dB, small", dB, 6);

    double P[5 * 4], Q[4 * 3], G[5 * 3], dP[5 * 4], dQ[4 * 3];
    fill(P, 20, 0.3);
    fill(Q, 12, 1.1);
    fill(G, 15, 2.5);
    matmul_backward(G, P, Q, dP, dQ, 5, 4, 3);
    show("dA, 5x4 times 4x3", dP, 20);
    show("dB, 5x4 times 4x3", dQ, 12);

    double only_dB[12], only_dA[20];
    matmul_backward(G, P, Q, NULL, only_dB, 5, 4, 3);     /* the gradient of A is not wanted */
    show("dB when dA is NULL", only_dB, 12);
    matmul_backward(G, P, Q, only_dA, NULL, 5, 4, 3);     /* the gradient of B is not wanted */
    show("dA when dB is NULL", only_dA, 20);
    return 0;
}
""", "the check")


matmul_backward_c = practice_native("matmul_backward", matmul_backward_c, matmul_backward_c_reference,
                                    matmul_backward_program)

**Expected behaviour.** ✅ once both gradients match. For the small case, $\bar A = \bar C B^\top$ gives `[1, -3, -0.5, 2, -1, 1.5]`. The last two cases pass `NULL` for one output; if your function writes through a `NULL` pointer, the check program crashes with a segmentation fault and the check says so. The next section confirms these formulas numerically, on the full network.

### ✍️ Your turn: `relu_backward` and `softmax_ce_backward`

Two short rules, in one cell:

- `relu_backward(Z, dH, dZ, n)`: `dZ[i]` is `dH[i]` where `Z[i] > 0`, and `0.0` otherwise. `Z` holds the layer's values *before* the ReLU.
- `softmax_ce_backward(logp, y, dZ, n, m)`: given the log-probabilities from `log_softmax_rows` and the labels `y`, `dZ[i * m + j]` becomes $(p_{ij} - [j = y_i]) / n$, where $p_{ij}$ is `exp(logp[i * m + j])`.

Hint: `(j == y[i] ? 1.0 : 0.0)` is the one-hot target, and dividing by `n` inside the loop keeps the gradient of the *mean* loss.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def output_rules_c_reference():
    return r"""
void relu_backward(const double *Z, const double *dH, double *dZ, long n)
{
    for (long i = 0; i < n; i++)
        dZ[i] = Z[i] > 0.0 ? dH[i] : 0.0;
}

void softmax_ce_backward(const double *logp, const int *y, double *dZ, long n, long m)
{
    for (long i = 0; i < n; i++)
        for (long j = 0; j < m; j++)
            dZ[i * m + j] = (exp(logp[i * m + j]) - (j == y[i] ? 1.0 : 0.0)) / n;
}
"""

In [ ]:
def output_rules_c():
    raise NotImplementedError  # ✍️ delete this line, then complete both C functions below
    return r"""
void relu_backward(const double *Z, const double *dH, double *dZ, long n)
{
    /* ✍️ Your code here (one loop) */
}

void softmax_ce_backward(const double *logp, const int *y, double *dZ, long n, long m)
{
    /* ✍️ Your code here (two loops) */
}
"""

In [ ]:
def output_rules_program(source):
    return CHECK_C + mark(source, "your relu_backward and softmax_ce_backward") + mark(r"""
int main(void)
{
    double Z[] = {-1.0, 0.0, 2.0, 1e-9, -1e-9}, dH[] = {5.0, 6.0, 7.0, 8.0, 9.0}, dZ[5];
    relu_backward(Z, dH, dZ, 5);
    show("relu_backward", dZ, 5);

    /* log-probabilities of three rows over two classes, labels 1, 0, 1 */
    double logp[] = {log(0.25), log(0.75), log(0.5), log(0.5), log(0.9), log(0.1)};
    int y[] = {1, 0, 1};
    double g[6];
    softmax_ce_backward(logp, y, g, 3, 2);
    show("softmax_ce_backward, 3 rows", g, 6);

    double row[] = {log(0.2), log(0.3), log(0.5)}, g1[3];
    int label = 2;
    softmax_ce_backward(row, &label, g1, 1, 3);
    show("one row of three classes", g1, 3);
    return 0;
}
""", "the check")


output_rules_c = practice_native("relu_backward and softmax_ce_backward", output_rules_c,
                                 output_rules_c_reference, output_rules_program)

**Expected behaviour.** `relu_backward` passes 7 and 8 through and blocks the rest, including the gradient at exactly 0. In the softmax case the first row, with $p = (0.25, 0.75)$ and label 1, gives $(0.25 - 0, 0.75 - 1)/3 = (0.0833, -0.0833)$: each row of the gradient sums to zero, and the true class gets a negative gradient, so gradient descent raises its logit.

## 7 · The network, and its gradients checked

The model is the 2 → 16 → 2 network from *Base Model Training*: $Z_1 = XW_1 + b_1$, $H = \operatorname{relu}(Z_1)$, $Z_2 = HW_2 + b_2$, and the mean cross-entropy of the log-softmax of $Z_2$. It has $2 \cdot 16 + 16 + 16 \cdot 2 + 2 = 82$ parameters.

In C, the model is a `struct`, a record of named fields, that holds the sizes and pointers to every buffer it needs. `mlp_new` allocates them all in **one block** with `calloc` (which zero-fills) and `mlp_free` releases it: in C, memory you allocate stays allocated until you free it. The four parameter arrays sit side by side in the block, followed by the four gradient arrays in the same order, so one loop can update every parameter, and Python can copy all 82 numbers in or out at once. The forward caches ($Z_1$, $H$, $Z_2$ and the log-probabilities) and the backward buffers are allocated for up to `capacity` rows.

Read the cell: it is the scaffold around the function you write next. `mlp_train_step` calls your `compute_gradients` and then takes one gradient-descent step, and `mlp_logits` runs the forward pass alone, in chunks of `capacity` rows.

In [ ]:
MLP_C = r"""
typedef struct {
    long d, h, c, capacity;          /* inputs, hidden units, classes, most rows per call */
    double *W1, *b1, *W2, *b2;       /* parameters: W1 is d x h, b1 is h, W2 is h x c, b2 is c */
    double *dW1, *db1, *dW2, *db2;   /* gradients of the mean loss, in the same order */
    double *Z1, *H1, *Z2, *logp;     /* forward caches, for up to capacity rows */
    double *dZ2, *dH1, *dZ1;         /* backward buffers, for up to capacity rows */
} Mlp;

long mlp_parameter_count(const Mlp *m)
{
    return m->d * m->h + m->h + m->h * m->c + m->c;
}

Mlp *mlp_new(long d, long h, long c, long capacity, const double *params)
{
    long p = d * h + h + h * c + c;
    Mlp *m = calloc(1, sizeof *m);
    double *block = calloc(2 * p + capacity * (4 * h + 3 * c), sizeof(double));
    if (!m || !block) {
        free(m);
        free(block);
        return NULL;
    }
    m->d = d; m->h = h; m->c = c; m->capacity = capacity;
    m->W1 = block;         m->b1 = m->W1 + d * h;    m->W2 = m->b1 + h;     m->b2 = m->W2 + h * c;
    m->dW1 = block + p;    m->db1 = m->dW1 + d * h;  m->dW2 = m->db1 + h;   m->db2 = m->dW2 + h * c;
    m->Z1 = block + 2 * p; m->H1 = m->Z1 + capacity * h;
    m->Z2 = m->H1 + capacity * h;   m->logp = m->Z2 + capacity * c;
    m->dZ2 = m->logp + capacity * c; m->dH1 = m->dZ2 + capacity * c; m->dZ1 = m->dH1 + capacity * h;
    memcpy(m->W1, params, p * sizeof(double));
    return m;
}

void mlp_free(Mlp *m)
{
    if (m) {
        free(m->W1);                 /* the start of the one block */
        free(m);
    }
}

void mlp_get(const Mlp *m, double *params, double *grads)
{
    long p = mlp_parameter_count(m);
    if (params) memcpy(params, m->W1, p * sizeof(double));
    if (grads) memcpy(grads, m->dW1, p * sizeof(double));
}

void mlp_set(Mlp *m, const double *params)
{
    memcpy(m->W1, params, mlp_parameter_count(m) * sizeof(double));
}

void bias_backward(const double *dZ, double *db, long n, long m)
{
    for (long j = 0; j < m; j++)
        db[j] = 0.0;
    for (long i = 0; i < n; i++)
        for (long j = 0; j < m; j++)
            db[j] += dZ[i * m + j];
}
"""

TRAIN_C = r"""
double mlp_train_step(Mlp *m, const double *X, const int *y, long n, double learning_rate)
{
    double loss = compute_gradients(m, X, y, n);
    long p = mlp_parameter_count(m);
    for (long i = 0; i < p; i++)     /* parameters and gradients are laid out in the same order */
        m->W1[i] -= learning_rate * m->dW1[i];
    return loss;
}

void mlp_logits(Mlp *m, const double *X, long n, double *out)
{
    for (long start = 0; start < n; start += m->capacity) {
        long rows = n - start < m->capacity ? n - start : m->capacity;
        matmul(X + start * m->d, m->W1, m->Z1, rows, m->d, m->h);
        add_bias(m->Z1, m->b1, rows, m->h);
        relu_forward(m->Z1, m->H1, rows * m->h);
        matmul(m->H1, m->W2, out + start * m->c, rows, m->h, m->c);
        add_bias(out + start * m->c, m->b2, rows, m->c);
    }
}
"""


def kernels_source():
    """Every kernel the model needs: yours where they passed their checks, otherwise the references."""
    return "".join([
        "#include <math.h>\n#include <stdlib.h>\n#include <string.h>\n",
        mark(relu_forward_c(), "relu_forward"), mark(matmul_c(), "matmul"),
        mark(ADD_BIAS_C, "add_bias"), mark(log_softmax_rows_c(), "log_softmax_rows"),
        mark(NLL_LOSS_C, "nll_loss"), mark(matmul_backward_c(), "matmul_backward"),
        mark(output_rules_c(), "relu_backward and softmax_ce_backward"), mark(MLP_C, "the Mlp scaffold"),
    ])


print("The scaffold is defined: 82 parameters for d = 2, h = 16, c = 2:", 2 * 16 + 16 + 16 * 2 + 2)

### ✍️ Your turn: `compute_gradients`

Complete `compute_gradients(m, X, y, n)`, the forward and backward pass of the whole network for a batch of `n` rows:

- **inputs:** `m`, the model; `X`, an $n \times d$ batch; `y`, the $n$ labels; `n`, at most `m->capacity`;
- **output:** the mean loss, as a `double`. As a side effect, `m->dW1`, `m->db1`, `m->dW2` and `m->db2` hold its gradients.

The forward half is written for you; write the backward half, which visits the same operations in reverse order:

1. `softmax_ce_backward` turns `m->logp` and `y` into `m->dZ2`;
2. `matmul_backward` of the second layer, $Z_2 = H W_2$, fills `m->dH1` and `m->dW2` (here $n \times k \times m$ is `n`, `m->h`, `m->c`);
3. `bias_backward(m->dZ2, m->db2, n, m->c)`;
4. `relu_backward` turns `m->dH1` into `m->dZ1`, using `m->Z1` from the forward pass, over all `n * m->h` elements;
5. `matmul_backward` of the first layer, $Z_1 = X W_1$, fills only `m->dW1`: pass `NULL` for the gradient of `X`;
6. `bias_backward(m->dZ1, m->db1, n, m->h)`.

`m->W2` is the field `W2` of the struct that `m` points to; `->` is how C reads a field through a pointer.

*Type it rather than pasting it: typing is how the order sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def compute_gradients_c_reference():
    return r"""
double compute_gradients(Mlp *m, const double *X, const int *y, long n)
{
    matmul(X, m->W1, m->Z1, n, m->d, m->h);
    add_bias(m->Z1, m->b1, n, m->h);
    relu_forward(m->Z1, m->H1, n * m->h);
    matmul(m->H1, m->W2, m->Z2, n, m->h, m->c);
    add_bias(m->Z2, m->b2, n, m->c);
    log_softmax_rows(m->Z2, m->logp, n, m->c);
    double loss = nll_loss(m->logp, y, n, m->c);

    softmax_ce_backward(m->logp, y, m->dZ2, n, m->c);
    matmul_backward(m->dZ2, m->H1, m->W2, m->dH1, m->dW2, n, m->h, m->c);
    bias_backward(m->dZ2, m->db2, n, m->c);
    relu_backward(m->Z1, m->dH1, m->dZ1, n * m->h);
    matmul_backward(m->dZ1, X, m->W1, NULL, m->dW1, n, m->d, m->h);
    bias_backward(m->dZ1, m->db1, n, m->h);
    return loss;
}
"""

In [ ]:
def compute_gradients_c():
    raise NotImplementedError  # ✍️ delete this line, then write the backward half below
    return r"""
double compute_gradients(Mlp *m, const double *X, const int *y, long n)
{
    matmul(X, m->W1, m->Z1, n, m->d, m->h);          /* forward: Z1 = X W1 + b1 */
    add_bias(m->Z1, m->b1, n, m->h);
    relu_forward(m->Z1, m->H1, n * m->h);            /* H = relu(Z1) */
    matmul(m->H1, m->W2, m->Z2, n, m->h, m->c);      /* Z2 = H W2 + b2 */
    add_bias(m->Z2, m->b2, n, m->c);
    log_softmax_rows(m->Z2, m->logp, n, m->c);
    double loss = nll_loss(m->logp, y, n, m->c);

    /* ✍️ Your code here: the six backward calls, in order */

    return loss;
}
"""

In [ ]:
def compute_gradients_program(source):
    return CHECK_C + kernels_source() + mark(source, "your compute_gradients") + mark(r"""
int main(void)
{
    double params[2 * 3 + 3 + 3 * 2 + 2], grads[17], X[5 * 2];
    int y[5] = {0, 1, 1, 0, 1};
    fill(params, 17, 0.4);
    fill(X, 10, 1.3);
    Mlp *m = mlp_new(2, 3, 2, 8, params);
    double loss = compute_gradients(m, X, y, 5);
    mlp_get(m, NULL, grads);
    show("loss, 5 rows", &loss, 1);
    show("gradients, 5 rows", grads, 17);

    loss = compute_gradients(m, X, y, 1);        /* a batch of one row */
    mlp_get(m, NULL, grads);
    show("loss, 1 row", &loss, 1);
    show("gradients, 1 row", grads, 17);
    mlp_free(m);
    return 0;
}
""", "the check")


compute_gradients_c = practice_native("compute_gradients", compute_gradients_c, compute_gradients_c_reference,
                                      compute_gradients_program)

**Expected behaviour.** ✅ once all six calls are present, in order, with the right buffers. Calling the two `matmul_backward`s in the wrong order still compiles but produces wrong gradients for the first layer, because `m->dZ1` would be read before `relu_backward` writes it. The next cell checks the result a second way, against central differences.

### Check the gradients against central differences

A gradient formula can be wrong in ways that still train a little. The reliable test perturbs each parameter $\theta_i$ by a small $\varepsilon$ and measures the change in the loss:

$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + \varepsilon) - L(\theta_i - \varepsilon)}{2 \varepsilon}.$$

It costs two forward passes per parameter, far too slow for training but ideal for testing. The cell loads the whole model as a library, declares every function's types, and compares the analytic gradients of all 82 parameters with the numerical ones on 50 rows of the training data. The relative error $\lVert g_{\text{num}} - g \rVert / (\lVert g_{\text{num}} \rVert + \lVert g \rVert)$ is about $10^{-9}$ or smaller when every rule is right, and near 1 when one is wrong.

In [ ]:
def quadrants(n, seed):
    """Points uniform in the square [-1, 1]^2; the label is 1 where x * y > 0 (top right and bottom left)."""
    gen = random.Random(seed)
    X, y = array("d"), array("i")                 # "i" is a C int, as nll_loss expects
    for _ in range(n):
        a, b = gen.uniform(-1, 1), gen.uniform(-1, 1)
        X.extend((a, b))
        y.append(1 if a * b > 0 else 0)
    return X, y


def he_init(sizes, seed):
    """Weights drawn from N(0, 2 / fan_in), biases zero, flattened as W1, b1, W2, b2."""
    gen = random.Random(seed)
    params = array("d")
    for fan_in, fan_out in zip(sizes[:-1], sizes[1:]):
        params.extend(gen.gauss(0.0, math.sqrt(2.0 / fan_in)) for _ in range(fan_in * fan_out))
        params.extend([0.0] * fan_out)
    return params


def load_model_library():
    lib = load(kernels_source() + mark(compute_gradients_c(), "compute_gradients") + mark(TRAIN_C, "train"))
    P, L, D = ctypes.c_void_p, ctypes.c_long, ctypes.c_double
    signatures = {
        "mlp_new": ([L, L, L, L, P], P),          # returns a pointer: restype must say so
        "mlp_free": ([P], None),
        "mlp_parameter_count": ([P], L),
        "mlp_get": ([P, P, P], None),
        "mlp_set": ([P, P], None),
        "compute_gradients": ([P, P, P, L], D),
        "mlp_train_step": ([P, P, P, L, D], D),
        "mlp_logits": ([P, P, L, P], None),
    }
    for name, (argtypes, restype) in signatures.items():
        getattr(lib, name).argtypes, getattr(lib, name).restype = argtypes, restype
    return lib


def norm(values):
    return math.sqrt(math.fsum(v * v for v in values))


X_train, y_train = quadrants(400, seed=10)
model_lib = load_model_library()
initial = he_init([2, 16, 2], seed=12)    # keep the array in a variable while C reads its memory
model = model_lib.mlp_new(2, 16, 2, 1000, address(initial))
P = model_lib.mlp_parameter_count(model)
params, analytic = array("d", [0.0]) * P, array("d", [0.0]) * P

model_lib.compute_gradients(model, address(X_train), address(y_train), 50)
model_lib.mlp_get(model, address(params), address(analytic))
numeric, eps = [], 1e-6
for i in range(P):
    saved = params[i]
    params[i] = saved + eps
    model_lib.mlp_set(model, address(params))
    plus = model_lib.compute_gradients(model, address(X_train), address(y_train), 50)
    params[i] = saved - eps
    model_lib.mlp_set(model, address(params))
    minus = model_lib.compute_gradients(model, address(X_train), address(y_train), 50)
    params[i] = saved
    numeric.append((plus - minus) / (2 * eps))
model_lib.mlp_set(model, address(params))

error = norm([a - b for a, b in zip(numeric, analytic)]) / max(norm(numeric) + norm(analytic), 1e-12)
print(f"{P} parameters; relative error between backward and central differences: {error:.1e}")
assert error < 1e-6, "a backward rule disagrees with central differences"

**Expected behaviour.** 82 parameters and a relative error around $10^{-9}$. Every gradient your C code computes agrees with the slow numerical one, so the backward pass is right. Notice that `mlp_new` was declared with `restype = c_void_p`: without it, `ctypes` would treat the returned pointer as a 32-bit `int`, and the program could crash on the next call.

## 8 · Put it together: train the network in C

Training is now a loop of `mlp_train_step` calls: each one runs your forward and backward pass on all 400 training points and takes one gradient-descent step with learning rate 0.5, all inside C. Python only starts the loop and records the loss. Afterwards, `mlp_logits` scores the 400 training points and 1,000 fresh points from the same distribution.

In [ ]:
model_lib.mlp_set(model, address(initial))   # start again from the same initial weights
X_fresh, y_fresh = quadrants(1000, seed=11)


def accuracy(X, y):
    logits = array("d", bytes(8 * 2 * len(y)))
    model_lib.mlp_logits(model, address(X), len(y), address(logits))
    return sum((logits[2 * i + 1] > logits[2 * i]) == (y[i] == 1) for i in range(len(y))) / len(y)


losses = []
start = time.perf_counter()
for step in range(300):
    losses.append(model_lib.mlp_train_step(model, address(X_train), address(y_train), 400, 0.5))
seconds = time.perf_counter() - start
train_accuracy, fresh_accuracy = accuracy(X_train, y_train), accuracy(X_fresh, y_fresh)
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f} in 300 steps, {seconds / 300 * 1e6:.0f} µs per step")
print(f"accuracy {train_accuracy:.1%} on the training points, {fresh_accuracy:.1%} on 1,000 fresh points")
assert losses[-1] < 0.5 * losses[0], "the training loss should at least halve"
assert train_accuracy >= 0.9 and fresh_accuracy >= 0.85, "the network should fit the quadrants task"
model_lib.mlp_free(model)          # C memory is freed by hand; the pointer must not be used again
print("All checks passed: your C code trains a network.")

**Expected behaviour.** The loss falls from about 0.69 to about 0.07, and the network classifies about 99% of both the training and the fresh points; on a laptop a step takes roughly 100 µs. These numbers describe a two-dimensional teaching task with no label noise. The last line frees the model: after `mlp_free`, the pointer in `model` refers to released memory, and using it again is a bug that C will not report.

## 9 · 🧪 Experiments: memory errors

C does not check array bounds. An index one past the end reads or writes whatever memory follows, which may belong to another array, so the program usually carries on with wrong numbers instead of stopping. **AddressSanitizer** is a compiler option (`-fsanitize=address`) that surrounds every allocation with guard bytes and checks every memory access, so the first out-of-bounds read stops the program with a report naming the line.

The cell below has a classic off-by-one bug: the inner loop runs `r <= k` instead of `r < k`, so each dot product reads one element too many, and the last ones read past the end of `B`. The same program runs three times: compiled without optimisation (`-O0`), with the usual optimisation (`-O2`), and with AddressSanitizer.

In [ ]:
BUGGY_PROGRAM = CHECK_C + mark(r"""
void matmul_buggy(const double *A, const double *B, double *C, long n, long k, long m)
{
    for (long i = 0; i < n; i++)
        for (long j = 0; j < m; j++) {
            double sum = 0.0;
            for (long r = 0; r <= k; r++)            /* the bug: <= reads one element too many */
                sum += A[i * k + r] * B[r * m + j];
            C[i * m + j] = sum;
        }
}
""", "matmul_buggy") + mark(r"""
int main(void)
{
    double *A = malloc(3 * 4 * sizeof(double)), *B = malloc(4 * 2 * sizeof(double));
    double *C = malloc(3 * 2 * sizeof(double));
    fill(A, 12, 0.1);
    fill(B, 8, 0.2);
    matmul_buggy(A, B, C, 3, 4, 2);
    show("C", C, 6);
    free(A); free(B); free(C);
    return 0;
}
""", "main")

for level in ("-O0", "-O2"):
    try:
        print(f"compiled with {level}:", run(build(BUGGY_PROGRAM, flags=(level,))).strip())
    except RunError as error:
        print(f"compiled with {level}, the program", error)

try:
    sanitized = build(BUGGY_PROGRAM, flags=("-fsanitize=address", "-g", "-O1"))
    report = run(sanitized, env={"ASAN_OPTIONS": "detect_leaks=0"})
    print("with AddressSanitizer it printed:", report.strip())
except CompileError as error:
    print("AddressSanitizer is not available with this compiler:", str(error).splitlines()[0])
except RunError as error:
    print("with AddressSanitizer:")
    shown = set()
    for line in str(error).splitlines():
        kind = next((key for key in ("ERROR: AddressSanitizer", "READ of", "WRITE of", "#0", "SUMMARY")
                     if key in line), None)
        if kind and kind not in shown:      # the first line of each kind: the error, the access, the frame
            shown.add(kind)
            print("  ", line.strip()[:140])

**What to notice.** Without the sanitizer, the program exits successfully and prints six numbers, and the two optimisation levels may even print different ones: some compilers print plausible values at `-O0` and `nan` at `-O2`. Reading past the end of an array is **undefined behaviour** in C, so the optimiser may assume it never happens and produce anything at all. With AddressSanitizer the program stops at the first bad read with `heap-buffer-overflow`, the size of the read (8 bytes, one `double`) and a stack frame in `matmul_buggy`. Compile with `-fsanitize=address -g -O1` whenever C code behaves strangely; higher optimisation can remove the very reads the sanitizer checks. It slows a program by about a factor of two, which is cheap for a test run. (On a few systems the sanitizer is not installed; the cell then says so.)

## Exercises

Three stretch challenges. Each has a check.

### Exercise 1: `matmul_fast`, the cache-friendly order

Write `matmul_fast(A, B, C, n, k, m)` with the loop order `i`, `r`, `j` from the experiment in section 4: clear `C` first, then add `A[i * k + r] * B[r * m + j]` into `C[i * m + j]` in the innermost loop. It must compute the same product as `matmul`; the check allows for the different order of the additions.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def matmul_fast_c_reference():
    return r"""
void matmul_fast(const double *A, const double *B, double *C, long n, long k, long m)
{
    for (long i = 0; i < n * m; i++)
        C[i] = 0.0;
    for (long i = 0; i < n; i++)
        for (long r = 0; r < k; r++) {
            double a = A[i * k + r];
            for (long j = 0; j < m; j++)
                C[i * m + j] += a * B[r * m + j];
        }
}
"""

In [ ]:
def matmul_fast_c():
    raise NotImplementedError  # ✍️ delete this line, then complete the C function below
    return r"""
void matmul_fast(const double *A, const double *B, double *C, long n, long k, long m)
{
    /* ✍️ Your code here: clear C, then loops in the order i, r, j */
}
"""

In [ ]:
def matmul_fast_program(source):
    return CHECK_C + mark(source, "your matmul_fast") + mark(r"""
int main(void)
{
    double P[6 * 7], Q[7 * 5], R[6 * 5];
    fill(P, 42, 0.5);
    fill(Q, 35, 1.5);
    for (int i = 0; i < 30; i++) R[i] = 123.0;   /* stale values that must be cleared */
    matmul_fast(P, Q, R, 6, 7, 5);
    show("6x7 times 7x5", R, 30);
    return 0;
}
""", "the check")


matmul_fast_c = practice_native("matmul_fast", matmul_fast_c, matmul_fast_c_reference, matmul_fast_program,
                                atol=1e-12, rtol=1e-12)

### Exercise 2: `leaky_relu_forward` and `leaky_relu_backward`

Leaky ReLU keeps a small slope $s$ for negative inputs, so a unit never stops passing gradient: $f(x) = x$ for $x > 0$ and $s x$ otherwise, with derivative 1 or $s$. Write both C functions, `leaky_relu_forward(x, y, n, slope)` and `leaky_relu_backward(x, dy, dx, n, slope)`, where `x` holds the inputs of the forward pass.

*Type it rather than pasting it: typing is how the syntax sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def leaky_relu_c_reference():
    return r"""
void leaky_relu_forward(const double *x, double *y, long n, double slope)
{
    for (long i = 0; i < n; i++)
        y[i] = x[i] > 0.0 ? x[i] : slope * x[i];
}

void leaky_relu_backward(const double *x, const double *dy, double *dx, long n, double slope)
{
    for (long i = 0; i < n; i++)
        dx[i] = x[i] > 0.0 ? dy[i] : slope * dy[i];
}
"""

In [ ]:
def leaky_relu_c():
    raise NotImplementedError  # ✍️ delete this line, then complete both C functions below
    return r"""
void leaky_relu_forward(const double *x, double *y, long n, double slope)
{
    /* ✍️ Your code here */
}

void leaky_relu_backward(const double *x, const double *dy, double *dx, long n, double slope)
{
    /* ✍️ Your code here */
}
"""

In [ ]:
def leaky_relu_program(source):
    return CHECK_C + mark(source, "your leaky_relu") + mark(r"""
int main(void)
{
    double x[] = {-3.0, -0.5, 0.0, 0.25, 4.0}, dy[] = {1.0, 2.0, 3.0, 4.0, 5.0}, y[5], dx[5];
    leaky_relu_forward(x, y, 5, 0.01);
    show("forward, slope 0.01", y, 5);
    leaky_relu_backward(x, dy, dx, 5, 0.01);
    show("backward, slope 0.01", dx, 5);
    leaky_relu_forward(x, y, 5, 0.2);
    show("forward, slope 0.2", y, 5);
    return 0;
}
""", "the check")


leaky_relu_c = practice_native("leaky_relu", leaky_relu_c, leaky_relu_c_reference, leaky_relu_program)

### Exercise 3: `count_correct`, evaluation in C

Write `long count_correct(const double *logits, const int *y, long n, long c)`, which returns how many rows have their largest logit at the true class. When two logits tie, the first (smallest index) counts as the prediction, the same rule as an `argmax`.

*Type it rather than pasting it: typing is how the syntax sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def count_correct_c_reference():
    return r"""
long count_correct(const double *logits, const int *y, long n, long c)
{
    long correct = 0;
    for (long i = 0; i < n; i++) {
        long best = 0;
        for (long j = 1; j < c; j++)
            if (logits[i * c + j] > logits[i * c + best])
                best = j;
        correct += best == y[i];
    }
    return correct;
}
"""

In [ ]:
def count_correct_c():
    raise NotImplementedError  # ✍️ delete this line, then complete the C function below
    return r"""
long count_correct(const double *logits, const int *y, long n, long c)
{
    /* ✍️ Your code here: an argmax per row, compared with y[i] */
}
"""

In [ ]:
def count_correct_program(source):
    return CHECK_C + mark(source, "your count_correct") + mark(r"""
int main(void)
{
    double logits[] = {0.1, 0.9, 0.0,
                       2.0, 1.0, 3.0,
                       0.5, 0.5, 0.1,     /* a tie: index 0 counts */
                       -1.0, -2.0, -0.5};
    int y[] = {1, 2, 1, 2};
    double counts[3];
    counts[0] = (double)count_correct(logits, y, 4, 3);
    counts[1] = (double)count_correct(logits, y, 2, 3);
    counts[2] = (double)count_correct(logits, y, 0, 3);
    show("correct rows of 4, of the first 2, of none", counts, 3);
    return 0;
}
""", "the check")


count_correct_c = practice_native("count_correct", count_correct_c, count_correct_c_reference, count_correct_program)

## Common errors and debugging

- **`error: expected ';' before '}' token`** or **`expected expression`.** A statement is missing its semicolon, or a brace or parenthesis is unbalanced. The compiler names your function and a line counted from its first line, such as `your matmul:6:13`; the real mistake is often on the line before.
- **`error: use of undeclared identifier 'k'`.** A variable or size is misspelt, or used outside the braces it was declared in. In C, a variable declared inside a loop's braces does not exist after the closing brace.
- **The check says ❌ with values that are almost right but larger.** An accumulator was not reset: `double sum = 0.0;` must be inside the loop over output elements, and buffers filled with `+=` must be cleared first.
- **Wrong answers only for non-square matrices.** An index uses the wrong size, such as `B[r * k + j]` instead of `B[r * m + j]`. Write each index as *row × row length + column* and check the row length against the matrix's shape.
- **`crashed with SIGSEGV`.** The program read or wrote memory it does not own: an index past the end, a `NULL` pointer, or a size argument that does not match the buffer. Run the program under AddressSanitizer, as in section 9, to find the line.
- **`nan` in the loss.** An overflowing `exp`, usually from a softmax without the maximum subtracted, or a `log` of zero.
- **Integer division.** In C, `1 / n` with an integer `n` is 0 for `n > 1`. Write `1.0 / n`, or divide a `double` by `n`.
- **A crash right after calling a C function from Python.** Check `argtypes` and `restype`: an undeclared `double` argument is passed as an integer, and a returned pointer without `restype = c_void_p` is cut to 32 bits.
- **Using memory after `free`.** The pointer still holds the old address, and reading through it may even seem to work. Set it aside and never use it again; AddressSanitizer reports such reads as `heap-use-after-free`.

## Summary

- A tensor is a contiguous block of memory plus a shape. In row-major order element $(i, j)$ of an $n \times m$ matrix is at offset $i m + j$, and strides describe views such as the transpose without moving data.
- Python calls compiled C through `ctypes`: compile into a shared library, declare `argtypes` and `restype`, and pass the address of an `array`, which C borrows only for the call.
- Matrix multiplication, a stable log-softmax and their gradients are short loops over raw memory. The order of the loops changes how well they use the cache, and therefore their speed, but not the result beyond rounding.
- The backward pass of a dense network is a fixed sequence of rules: $\bar A = \bar C B^\top$ and $\bar B = A^\top \bar C$, column sums for biases, a mask for ReLU, and $(p - \text{onehot}(y)) / n$ for softmax cross-entropy. Central differences confirm every one.
- C does not check bounds or free memory for you. AddressSanitizer finds out-of-bounds and use-after-free errors that otherwise give silently wrong numbers.

Notebook 2 moves to C++ and replaces the hand-ordered backward calls with an engine that records the forward pass and derives the backward pass itself, with memory that frees itself.